# 🧪 Mental Loop: Simulate an Action Before Taking It

## Learning Objectives
In this notebook, you will learn:
1. **The mental-model-in-the-loop idea** - Test a proposed action in a copy of the world before acting in the real one
2. **A simulator environment** - A small market model that can be forked, stepped and rolled out many times
3. **Propose → simulate → refine → execute** - The loop as a LangGraph graph, with the LLM proposing and code checking risk
4. **Seeing the refinement happen** - The same proposal is accepted in a calm market and cut down in a turbulent one

## Where this fits
Notebooks `01`–`03` searched over *reasoning steps*. Here the agent searches over *actions in an environment*: before it trades, it runs the proposed trade through hundreds of simulated futures and only executes a version whose worst cases are acceptable. The planning agents of module 10 act first and evaluate the result afterwards (`modules/10_planning/notebooks/02_planning_decompose_execute_replan.ipynb`); a simulator lets the agent evaluate **before** the action is irreversible.

## Prerequisites
- LangGraph graphs and structured output - `modules/10_planning/notebooks/01_planning_react_vs_plan.ipynb`
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`

| Property | Value |
|---|---|
| Origin | Robotics "world models" + classical AI deliberation (model-based planning) |

> This is a teaching toy, not trading advice: the "market" is a random walk with a news-dependent drift.

---
## 📦 Step 1: Environment Setup

Load the API keys from the project-root `.env`. LangSmith tracing is optional: it is used only if your `.env` turns it on.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import os
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

# Optional: LangSmith traces every LLM call if the .env sets LANGSMITH_TRACING=true.
# Nothing in this notebook depends on it.
tracing = os.getenv("LANGSMITH_TRACING", os.getenv("LANGCHAIN_TRACING_V2", "false")).lower() == "true"
print(f"✅ Environment ready (LangSmith tracing: {'on' if tracing else 'off'})")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: The analyst that proposes trades
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

---
## 📖 Part 1: The Mental-Model-in-the-Loop Idea

The core idea is to give an agent the ability to "think before it acts" in a very concrete way. Instead of committing to an action in the real world immediately, the agent first tests its proposed action in an internal, simulated version of the environment. By observing the likely consequences in this safe sandbox, it can evaluate risks, refine its strategy, and only then execute a more considered action in reality.

### High-level Workflow
1. **Observe** the current state of the real environment.
2. **Propose** an action (here: an LLM analyst proposes a trade).
3. **Simulate**: fork the environment and roll the action forward many times to see the range of outcomes.
4. **Refine**: keep the action if its outcomes are acceptable, otherwise shrink it (or do nothing).
5. **Execute** the refined action in the *real* environment.

### When to Use / Applications
*   **Robotics:** simulating a grasp or a path before moving a physical arm.
*   **High-stakes decisions:** simulating a trade's impact on a portfolio, or a change's impact on a system, before committing.
*   **Games:** simulating several moves ahead before choosing one.

### Strengths & Weaknesses
*   **Strengths:** catches costly mistakes while they are still hypothetical; turns "is this a good idea?" into numbers you can put a rule on.
*   **Weaknesses:** the **simulation-reality gap** — decisions are only as good as the model of the world; and running many rollouts costs time.

---
## 🌍 Part 2: The Simulator Environment

`MarketSimulator` holds a price, a news headline, a portfolio, and a **volatility** (daily standard deviation of returns). Each `step` moves the price by a random daily return whose average depends on the news. Two design points matter for the pattern:

- The same class is the "real world" and the sandbox: `model_copy(deep=True)` forks it, so a simulation can never touch the real portfolio.
- Randomness comes from an explicit `numpy` generator, so the simulations are reproducible and the real market uses a different stream than the agent's rollouts.

The analyst only reads the headline, price and portfolio (`describe()`); the volatility is part of the simulator's model of the world.

In [ ]:
# ============================================================================
# MARKET SIMULATOR: one stock, one portfolio, news-driven drift
# ============================================================================
import numpy as np
from pydantic import BaseModel

NEWS_DRIFT = {  # average daily return implied by each headline
    "Market is stable.": 0.0,
    "Positive earnings report expected.": 0.01,
    "New competitor enters the market.": -0.01,
}


class MarketSimulator(BaseModel):
    day: int = 0
    price: float = 100.0
    volatility: float = 0.02
    news: str = "Market is stable."
    cash: float = 10_000.0
    shares: int = 0

    def value(self) -> float:
        return self.cash + self.shares * self.price

    def trade(self, action: str, shares: int) -> int:
        """Execute a trade, clamped to what the portfolio can afford; returns shares traded."""
        if action == "buy":
            shares = min(shares, int(self.cash // self.price))
            self.cash -= shares * self.price
            self.shares += shares
        elif action == "sell":
            shares = min(shares, self.shares)
            self.cash += shares * self.price
            self.shares -= shares
        else:
            shares = 0
        return shares

    def step(self, rng: np.random.Generator) -> None:
        self.price *= 1 + rng.normal(NEWS_DRIFT[self.news], self.volatility)
        self.day += 1

    def describe(self) -> str:
        return (f"Day {self.day}: price ${self.price:.2f}. News: {self.news} "
                f"Portfolio: {self.shares} shares, ${self.cash:,.2f} cash (total ${self.value():,.2f}).")


demo = MarketSimulator()
fork = demo.model_copy(deep=True)
fork.trade("buy", 10)
print("real :", demo.describe())
print("fork :", fork.describe())

In [ ]:
# ============================================================================
# ROLLOUTS: fork the market, apply an action, simulate N futures
# ============================================================================
N_ROLLOUTS = 200
HORIZON_DAYS = 10


def simulate(market: MarketSimulator, action: str, shares: int, seed: int = 0) -> dict:
    """Portfolio return (%) after HORIZON_DAYS, over N_ROLLOUTS forked futures."""
    rng = np.random.default_rng(seed)
    returns = []
    for _ in range(N_ROLLOUTS):
        sandbox = market.model_copy(deep=True)       # the real market is never touched
        start_value = sandbox.value()
        sandbox.trade(action, shares)
        for _ in range(HORIZON_DAYS):
            sandbox.step(rng)
        returns.append((sandbox.value() - start_value) / start_value * 100)
    r = np.array(returns)
    return {"mean": r.mean(), "p5": np.percentile(r, 5), "p_loss": (r < 0).mean() * 100}


calm = MarketSimulator(news="Positive earnings report expected.", volatility=0.02)
stats = simulate(calm, "buy", 50)
print(f"buy 50 shares in a calm market -> mean {stats['mean']:+.2f}%, "
      f"5th percentile {stats['p5']:+.2f}%, P(loss) {stats['p_loss']:.0f}%")

The **5th percentile** is the number to watch: in 1 simulated future out of 20, the portfolio does at least this badly. That is the "how bad can it get?" signal a real risk desk uses, and it is what the refine step checks.

---
## 🔁 Part 3: Propose → Simulate → Refine → Execute

- **`propose`** (LLM): the analyst reads `describe()` and returns a structured proposal — `buy`/`sell`/`hold` and a `small` (10%) or `large` (50%) size — which code turns into a share count. No keyword matching on free text.
- **`simulate`** (code): rolls out the proposal and progressively **safer** versions of it — a smaller buy (½, ¼, ⅛, then nothing), or a larger sell (×2, ×4, then everything).
- **`refine`** (code): keeps the version **closest to the proposal** whose 5th-percentile return is no worse than `MAX_P5_LOSS`; if none passes, the safest one. A numeric rule belongs in code: while prototyping, an LLM "risk manager" given this rule in its prompt answered *hold* even when the proposal passed it.
- **`execute`**: trades in the real market, then advances it one day with the real market's own randomness.

The graph builder takes the proposer as an argument, so Part 4 can also run it with a stub proposer and no LLM.

In [ ]:
# ============================================================================
# SCHEMAS, NODES AND GRAPH BUILDER
# ============================================================================
from typing import Callable, Literal, Optional, TypedDict

from langgraph.graph import END, START, StateGraph
from pydantic import Field

MAX_P5_LOSS = -5.0                       # risk rule: 5th-percentile 10-day return must be >= -5%
SIZE_FRACTION = {"small": 0.10, "large": 0.50}


class Proposal(BaseModel):
    action: Literal["buy", "sell", "hold"]
    size: Literal["small", "large"] = Field(description="small = 10% of cash (buy) or of shares (sell); large = 50%")
    reasoning: str = Field(description="One sentence")


class AgentState(TypedDict):
    market: MarketSimulator
    proposal: Optional[Proposal]
    proposed_shares: int
    simulations: list
    decision: Optional[dict]


def llm_proposer(market: MarketSimulator) -> Proposal:
    return llm.with_structured_output(Proposal).invoke(
        f"You are a trading analyst. Based on the market state, propose one trade.\n\n{market.describe()}"
    )


def to_shares(market: MarketSimulator, p: Proposal) -> int:
    fraction = SIZE_FRACTION[p.size]
    if p.action == "buy":
        return int(market.cash * fraction // market.price)
    if p.action == "sell":
        return int(market.shares * fraction)
    return 0


def candidate_trades(market: MarketSimulator, action: str, shares: int) -> list:
    """The proposal first, then progressively safer versions of it (duplicates removed)."""
    if action == "buy":
        candidates = [("buy", int(shares * k)) for k in (1, 0.5, 0.25, 0.125)] + [("hold", 0)]
    elif action == "sell":
        candidates = [("sell", min(market.shares, shares * k)) for k in (1, 2, 4)] + [("sell", market.shares)]
    else:
        candidates = [("hold", 0), ("sell", market.shares // 2), ("sell", market.shares)]
    unique = []
    for a, n in candidates:
        a = a if n else "hold"
        if (a, n) not in unique:
            unique.append((a, n))
    return unique


def build_agent(proposer: Callable[[MarketSimulator], Proposal], real_rng: np.random.Generator):
    def propose_node(state: AgentState) -> dict:
        p = proposer(state["market"])
        shares = to_shares(state["market"], p)
        print(f"🧐 PROPOSE : {p.action} {p.size} ({shares} shares) - {p.reasoning}")
        return {"proposal": p, "proposed_shares": shares}

    def simulate_node(state: AgentState) -> dict:
        p, market = state["proposal"], state["market"]
        sims = []
        for action, shares in candidate_trades(market, p.action, state["proposed_shares"]):
            stats = simulate(market, action, shares)
            sims.append({"action": action, "shares": shares, **stats})
            print(f"🤖 SIMULATE: {action:<4} {shares:>3} shares -> mean {stats['mean']:+6.2f}%  "
                  f"p5 {stats['p5']:+6.2f}%  P(loss) {stats['p_loss']:3.0f}%")
        return {"simulations": sims}

    def refine_node(state: AgentState) -> dict:
        # Candidates are ordered from "as proposed" to "safest": take the first that passes the rule,
        # or the safest one if none does.
        passing = [s for s in state["simulations"] if s["p5"] >= MAX_P5_LOSS]
        decision = passing[0] if passing else state["simulations"][-1]
        changed = (decision["action"], decision["shares"]) != (state["proposal"].action, state["proposed_shares"])
        print(f"🧠 REFINE  : {decision['action']} {decision['shares']} shares "
              f"({'changed to pass' if changed else 'proposal passes'} the p5 >= {MAX_P5_LOSS}% rule)")
        return {"decision": decision}

    def execute_node(state: AgentState) -> dict:
        market = state["market"].model_copy(deep=True)
        traded = market.trade(state["decision"]["action"], state["decision"]["shares"])
        market.step(real_rng)
        print(f"🚀 EXECUTE : {state['decision']['action']} {traded} shares -> {market.describe()}")
        return {"market": market}

    builder = StateGraph(AgentState)
    builder.add_node("propose", propose_node)
    builder.add_node("simulate", simulate_node)
    builder.add_node("refine", refine_node)
    builder.add_node("execute", execute_node)
    builder.add_edge(START, "propose")
    builder.add_edge("propose", "simulate")
    builder.add_edge("simulate", "refine")
    builder.add_edge("refine", "execute")
    builder.add_edge("execute", END)
    return builder.compile()


agent = build_agent(llm_proposer, real_rng=np.random.default_rng(42))
print("🧩 Nodes:", [n for n in agent.get_graph().nodes if not n.startswith("__")])

---
## 📊 Part 4: The Same Proposal in Two Different Worlds

Scenarios A and B look **identical to the analyst**: same price, same good-news headline, same portfolio. They differ only in volatility, which only the simulator knows about. If the mental loop works, the proposal survives in the calm market and is cut down in the turbulent one. Scenario C holds shares on bad news.

In [ ]:
# ============================================================================
# SCENARIOS A-C: calm good news, turbulent good news, bad news while invested
# ============================================================================
scenarios = {
    "A: calm, good news (vol 2%)": MarketSimulator(news="Positive earnings report expected.", volatility=0.02),
    "B: turbulent, good news (vol 8%)": MarketSimulator(news="Positive earnings report expected.", volatility=0.08),
    "C: bad news, 40 shares held (vol 3%)": MarketSimulator(news="New competitor enters the market.", volatility=0.03,
                                                             cash=6_000.0, shares=40),
}

results = {}
for name, market in scenarios.items():
    print(f"\n=== {name} ===")
    print("   analyst sees:", market.describe())
    results[name] = agent.invoke({"market": market})

In [ ]:
# ============================================================================
# WHAT DID THE SIMULATOR CHANGE?
# ============================================================================
print(f"{'scenario':<38}{'proposed':>18}{'executed':>18}")
for name, r in results.items():
    proposed = f"{r['proposal'].action} {r['proposed_shares']}"
    executed = f"{r['decision']['action']} {r['decision']['shares']}"
    print(f"{name:<38}{proposed:>18}{executed:>18}")

Read the two tables together. In the calm market the full proposal already meets the rule, so the loop costs time but changes nothing. In the turbulent market the same headline would have put half the cash at risk with a 5th-percentile loss far beyond the rule; the simulator shows it and the refine step shrinks the trade until it passes. In scenario C the safer direction is the opposite one — selling *more* — which is why the candidates depend on the action. A naive agent without the loop would have executed the proposal as written in both worlds.

The analyst is an LLM, so its proposal can vary between runs. To prove the refine logic itself, the next cell swaps in a **stub proposer** that always proposes the same large buy, with no LLM call.

In [ ]:
# ============================================================================
# STUB PROPOSER: prove the refine rule without an LLM
# ============================================================================
def always_buy_large(market: MarketSimulator) -> Proposal:
    return Proposal(action="buy", size="large", reasoning="stub: always buy large")


stub_agent = build_agent(always_buy_large, real_rng=np.random.default_rng(0))
calm_run = stub_agent.invoke({"market": scenarios["A: calm, good news (vol 2%)"]})
turbulent_run = stub_agent.invoke({"market": scenarios["B: turbulent, good news (vol 8%)"]})

assert calm_run["decision"]["shares"] == calm_run["proposed_shares"], "calm market: proposal should pass unchanged"
assert turbulent_run["decision"]["shares"] < turbulent_run["proposed_shares"], "turbulent market: proposal must be reduced"
print("\n✅ Same proposal: accepted in the calm market, reduced in the turbulent market")

### The simulation-reality gap
Everything above is only as good as `MarketSimulator`. If the real market's volatility were 8% while the simulator assumed 2%, the loop would approve the risky trade with full confidence. In practice the simulator's parameters must be estimated from data and re-checked, and the rule should leave a margin for model error.

---
## 📝 Summary

In this notebook, we learned:

### 1. The mental-model-in-the-loop idea
- Evaluate an action in a forked copy of the world before it becomes irreversible.

### 2. A simulator environment
- One class serves as both the real world and the sandbox; `model_copy(deep=True)` forks it and a seeded generator makes rollouts reproducible.

### 3. Propose → simulate → refine → execute
- The LLM proposes a structured action; the simulator rolls out the proposal and safer variants; a numeric risk rule, enforced in code, picks the acceptable version closest to the proposal.

### 4. Seeing the refinement happen
- The same proposal passes in a calm market and is reduced in a turbulent one; a stub proposer proves the rule without any LLM call. The loop is only as good as the simulator's model of the world.

### Next Steps
- **`05_reflexive_metacognitive.ipynb`**: instead of simulating the world, the agent reasons about *itself* — its knowledge domain, tools and limits — and decides whether to answer, use a tool or escalate